# Cat bond fund analysis

This analyzes a diverse range of cat bond funds with publicly available data.

## Sources

Data was gathered from various public sources. Links to sufficient sources are provided in the headers of the raw CSVs ([link](../data/raw/)) for reference and verification.

## Setup

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from analysis.data import checks
from analysis.data.panels import build_panels
from analysis.data.sources import load_sources
from analysis.data.store import CsvStore
from analysis.paths import FIGURES_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR
from analysis.plotting import plots
from analysis.plotting.figure import save
from analysis.utils import expect_columns

raw = CsvStore(RAW_DATA_DIR)
processed = CsvStore(PROCESSED_DATA_DIR)
sources = load_sources(raw)

## Check raw data

Before processing, compound the raw monthly returns within each calendar year and compare them with the reported annual returns. Both inputs are the unadjusted series from the raw CSVs. Only fund-years with all twelve months are included. The tolerance is derived from the quoting precision in `checks.TOL_PP`; larger gaps point to likely data errors. The plot is shown inline for spot-checking.

In [ ]:
def check_raw(tol: float = checks.TOL_PP) -> None:
    monthly = sources.fund_monthly.data
    yearly = sources.fund_yearly.data
    deviations = checks.annual_deviations(monthly, yearly)
    summary, flags = checks.summarize(deviations, tol=tol)

    print("Flags above tolerance (pp):")
    display(flags)
    print("Per-fund deviation summary (pp):")
    display(summary.round(3))

    plots.annual_deviations(deviations, tol=tol)


check_raw()

## Process returns

Build the named return panels used for analysis, and persist them.

In [ ]:
panels = build_panels(sources, processed, save=True)
print("Processed panels:\n- " + "\n- ".join(panels))

## Explore

In [ ]:
SELECTED_FUNDS = (
    "Fermat UCITS Cat Bond Fund",
    "Franklin Cat Bond UCITS Fund",
    "Leadenhall UCITS ILS Fund",
)
SELECTED_INDEXES = (
    "Plenum CAT Bond UCITS Fund Index - Master Capital",
    "With Intelligence ILS Index - USD Hedged",
)

selected = (*SELECTED_FUNDS, *SELECTED_INDEXES)
cat_bond = (*panels["cash_excess"].columns, *SELECTED_INDEXES)
excess_returns = expect_columns(panels["excess_returns"], cat_bond)
total_returns = expect_columns(panels["total_returns"], selected)


def explore_returns() -> None:
    heatmap, _ = plots.monthly_heatmap(
        excess_returns,
        columns=cat_bond,
        months=36,
        title="Monthly returns relative to USD cash",
    )
    save(heatmap, FIGURES_DIR / "heatmap.png")

    plots.growth(
        excess_returns,
        columns=selected,
        months=24,
        title="Cumulative performance relative to USD cash",
    )

    plots.drawdowns(
        total_returns,
        columns=selected,
        months=36,
    )

    plots.rolling_returns(
        excess_returns,
        columns=selected,
        window=12,
        display_months=36,
    )


explore_returns()


![heatmap](figures/heatmap.png)